In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/vietnam_real_estate_fnl.csv")

# 11. Vị trí nhà (mặt tiền/hẻm...) ảnh hưởng thế nào đến giá?

với bộ dữ liệu này chưa có cột location_type(mặt tiền), nên giờ phải tạo thêm 1 cột mới được trích xuất từ cột name và description

In [30]:
# gộp 2 cột name và description lại để tìm thông tin mặt tiền, hẹp từ 2 cột này
df["text_content"] = (
    df["name"].fillna("") + " " + 
    df["description"].fillna("")
).str.lower()


In [ ]:
df["text_content"]

0       bán đất thanh đàm, ô tô vào nhà, cách phố 15m,...
1       lô góc kđt bắc châu giang, 97m2, chân 34 tòa c...
2       siêu rẻ. bán đất diện tích 105m2, mặt đường ph...
3       bán tòa chdv nguyễn thị minh khai, q1 - tn: 72...
4       bán đất tặng nhà 2 tầng mặt tiền đường lê quan...
                              ...                        
9995    mỹ đình plaza 1, bán căn hộ 3pn, full nội thất...
9996    nhà khu long trường quận 9, đầy đủ công năng, ...
9997    nhà mới tinh phan bá phiến, phường 7 tân bình,...
9998    chủ ngộp bank gửi bán gấp lô đất 318m2 full th...
9999    căn duy nhất - khu nhà cư xá phú lâm b nhà cư ...
Name: text_location, Length: 10000, dtype: object

In [ ]:
#phân loại mặt tiền/hẻm

def classify_location(text):
    text = str(text).lower()
    
    frontage_keywords = [
        "mặt tiền",
        "mặt đường",
        "mặt phố"
    ]
    
    alley_keywords = [
        "trong hẻm",
        "hẻm",
        "trong ngõ",
        "ngõ",
        "kiệt"
    ]
    
    # Kiểm tra mặt tiền trước
    if any(keyword in text for keyword in frontage_keywords):
        return "Mặt tiền"
    
    elif any(keyword in text for keyword in alley_keywords):
        return "Hẻm/Ngõ"
    
    else:
        return "Không xác định"

df["location_type"] = df["text_content"].apply(classify_location)

In [15]:
df["location_type"] == "Mặt tiền"

0       False
1        True
2        True
3       False
4        True
        ...  
9995    False
9996    False
9997    False
9998    False
9999    False
Name: location_type, Length: 10000, dtype: bool

In [16]:
#kiểm tra số lượng 
df["location_type"].value_counts(dropna=False)

location_type
Không xác định    4345
Mặt tiền          3917
Hẻm/Ngõ           1738
Name: count, dtype: int64

với cột trên thì ta đã  tách biệt được ví trị của bất động sản là ở mặt tiền hay ở hẻm.

In [ ]:
# hàm này dùng để xuất ra các mẫu có loại là nhà mặt tiền để ta kiểm tra thủ công 
# xem thử nó có phải là nhà mặt tiền đúng không hay là như: 
# kiểu (nhà cách mặt tiền 100m) -> thực chất nhà này không nằm ở mặt tiền mà nằm trong góc khác mà ở 
# mô tả lại có chữ "mặt tiền" nên nó được xếp vào loại là "Mặt tiền"
df.loc[
    df["location_type"] == "Mặt tiền",
    ["name", "description", "location_type"]
].sample(10, random_state=42)

,name,description,location_type
674,"Nhà HXH, 65m2, 4,5x15,2t, 6,4 tỷ,Nguyễn Văn Kh...",Nhà riêng 2 tầng siêu xinh nằm ở Nguyễn Văn Kh...,Mặt tiền
5227,"Bán đất ở biển ở TDP Hòa Bình, Nam Định 79m2, ...","Bán đất ở biển thổ cư chính chủ TDP Hòa Bình, ...",Mặt tiền
2071,Nhà xây mới 5 tầng ngay chợ Bà Chiểu xe hơi đỗ...,Nhà xây mới hoàn toàn 5 tầng BTCT chắc chắn gồ...,Mặt tiền
7446,Bán nhà Gốc Đề. Lô góc 2 thoáng vĩnh viễn mặt ...,Cơ hội hiếm! Sở hữu nhà trung tâm Gốc Đề giá h...,Mặt tiền
5206,"Tuyệt phẩm đất nền Minh Khai, Bắc Giang vị trí...",Bừng sáng cơ hội sở hữu 3 lô đất liền kề tại M...,Mặt tiền
9437,Bán đất mặt tiền 10m cực hiếm ngay phố KD Nguy...,Bán đất phố kinh doanh Nguyễn Hữu Thọ đoạn sầm...,Mặt tiền
8957,Bán nhà Hoàng Quốc Việt - 40m2 x 5 tầng x mt 7...,Do chuyển công tác cần bán gấp nhà phố Hoàng S...,Mặt tiền
5949,"Bán Đất cạnh Chill Villa tại Yên Bài, Ba Vì, H...","Đất nền cực chất tại Xã Yên Bài, Ba Vì, Hà Nội...",Mặt tiền
4718,"Tuyệt phẩm, bán biệt thự An Quý Villa Dương Nộ...","Tuyệt phẩm, bán biệt thự An Quý Villa Dương Nộ...",Mặt tiền
4502,"Lê Văn Sỹ, Quận 3, vị trí Kd, 5 Tầng, 49m2, hẻ...","""Lê Văn Sỹ, Quận 3, vị trí Kd, 5 Tầng, 49m2, h...",Mặt tiền


để so sánh trực quan hơn thì nên so sánh theo giá tiền trên m2. nên tạo thêm 1 cột giá 1m2 sẽ là bao nhiêu. với cột price_per_m2 = price/area

In [19]:
df["price_per_m2"] = df["price"] / df["area"]

In [20]:
# đổi sang triệu đồng /m2
df["price_per_m2_million"] = (
    df["price"] / df["area"] / 1_000_000
)

In [ ]:
df["price_per_m2_million"]

0       155.000000
1              NaN
2       185.714286
3       263.888889
4       359.375000
           ...    
9995     84.745763
9996     75.000000
9997           NaN
9998      2.830189
9999    197.500000
Name: price_per_m2_million, Length: 10000, dtype: float64

In [25]:
location_price_stats = (
    df.groupby("location_type")
      .agg(
          count=("price", "count"),
          median_price=("price", "median"),
          mean_price=("price", "mean"),
          median_price_per_m2=("price_per_m2_million", "median"),
          mean_price_per_m2=("price_per_m2_million", "mean"),
          median_area=("area", "median")
      )
      .sort_values("median_price_per_m2", ascending=False)
)

location_price_stats

,count,median_price,mean_price,median_price_per_m2,mean_price_per_m2,median_area
location_type,,,,,,
Hẻm/Ngõ,1632,7.375000e+09,9.376539e+09,132.000000,152.693544,58.0
Mặt tiền,3677,9.100000e+09,1.949395e+10,125.806452,162.035967,88.0
Không xác định,3918,6.000000e+09,1.095055e+10,74.324324,101.573812,81.0


In [27]:
location_price_stats = (
    df.groupby("location_type")
      .agg(
          count=("price", "count"),
          mean_price=("price", "mean"),
          mean_price_per_m2=("price_per_m2_million", "mean"),
          mean_area=("area", "mean")
      )
      .sort_values("mean_price_per_m2", ascending=False)
)

location_price_stats

,count,mean_price,mean_price_per_m2,mean_area
location_type,,,,
Mặt tiền,3677,1.949395e+10,162.035967,276.265297
Hẻm/Ngõ,1632,9.376539e+09,152.693544,77.117791
Không xác định,3918,1.095055e+10,101.573812,202.517429


từ kết quả trên cho thấy là vị trí có mối liên hệ với giá bất động sản. Khu bắt động sản Mặt tiền có giá trung bình cao nhất, đạt khoảng 19,49 tỷ đồng, cao hơn khoảng 108% so với nhóm Hẻm/Ngõ.  Nhưng khi chuẩn hóa theo diện tích, giá trung bình trên mỗi mét vuông của nhóm Mặt tiền đạt khoảng 162,04 triệu đồng/m², cao hơn khoảng 6,1% so với nhóm Hẻm/Ngõ, đạt 152,69 triệu đồng/m². Kết quả này cho thấy giá trị bất động sản mặt tiền vẫn cao hơn ở trong hẽm/ngõ nhưng mà do diện tích trung bình của nhóm Mặt tiền cũng lớn hơn nhiều so với hẻm ngõ (276,27 m² so với 77,12 m²). Do đó, chưa thể kết luận toàn bộ sự khác biệt về giá là do vị trí của bđs, vì giá còn có thể chịu ảnh hưởng bởi diện tích và các yếu tố khác

# 12. Tình trạng pháp lý ảnh hưởng thế nào đến giá?

In [31]:
# ta cần tách các loại ra như là có sổ đỏ, sổ hồng, pháp lý đầy đủ,... 
# và các loại như chờ sổ,... ra riêng để phân tích
def classify_legal_status(text):
    text = str(text).lower()

    # Nhóm đã có giấy tờ pháp lý rõ ràng
    legal_keywords = [
        "sổ đỏ",
        "sổ hồng",
        "đã có sổ",
        "sổ riêng",
        "sổ chính chủ",
        "giấy chứng nhận quyền sử dụng đất",
        "gcn quyền sử dụng đất",
        "gcnqsdđ",
        "pháp lý đầy đủ",
        "pháp lý rõ ràng",
        "công chứng sang tên"
    ]

    # Nhóm pháp lý chưa hoàn thiện
    pending_keywords = [
        "đang làm sổ",
        "chờ sổ",
        "chưa có sổ",
        "chưa ra sổ",
        "đợi sổ"
    ]

    if any(keyword in text for keyword in pending_keywords):
        return "Pháp lý chưa hoàn thiện"

    elif any(keyword in text for keyword in legal_keywords):
        return "Có sổ/Giấy tờ"

    else:
        return "Không đề cập"

df["legal_status"] = df["text_content"].apply(classify_legal_status)

In [32]:
df["legal_status"].value_counts()

legal_status
Có sổ/Giấy tờ              5202
Không đề cập               4776
Pháp lý chưa hoàn thiện      22
Name: count, dtype: int64

In [33]:
legal_price_stats = (
    df.groupby("legal_status")
      .agg(
          count=("price", "count"),
          mean_price=("price", "mean"),
          mean_price_per_m2=(
              "price_per_m2_million",
              "mean"
          ),
          mean_area=("area", "mean")
      )
      .sort_values("mean_price_per_m2", ascending=False)
)

legal_price_stats

,count,mean_price,mean_price_per_m2,mean_area
legal_status,,,,
Có sổ/Giấy tờ,4823,1.381182e+10,144.191282,214.190340
Không đề cập,4383,1.439507e+10,124.543897,205.077883
Pháp lý chưa hoàn thiện,21,8.476952e+09,78.945614,110.454545


từ kết quả trên thì ta thấy giá trị trung bình của bds có giấy tờ thấp hơn 1 ít so với không có đề cập. nhưng giá trị trung bình /m2 lại cao hơn cho thấy những bds có pháp lý đầy đủ thì giá nó vẫn sẽ cao hơn so với những bds khác. mà về trạng thái không đề cập thì chưa chắc đã không có giấy tờ hoặc là đã có giấy tờ đầy đủ, vì đây chỉ là thông tin từ người bán đề cập tới. Còn với 1 số loại bds chưa hoàn thiện giấy tờ thì thấy rõ được giá thấp hơn nhiều so với các loại khác tuy nhiên với số lượng quá ít so với 2 cái kia nên chưa thể đánh giá 1 cách trực quan hơn được.

# 13. Hướng nhà ảnh hưởng thế nào đến giá?

In [34]:
# xem các hướng của bds 
df["house_direction"].value_counts(dropna=False)

house_direction
NaN           6949
Đông Nam       403
Đông           325
Nam            322
Đông - Nam     295
Đông Bắc       256
Tây Bắc        237
Tây Nam        234
Tây - Bắc      232
Bắc            231
Đông - Bắc     188
Tây - Nam      168
Tây            160
Name: count, dtype: int64

In [36]:
# chuẩn hóa dữ liệu trươc (bỏ dấu - )
df["house_direction_clean"] = (
    df["house_direction"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s*-\s*", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [38]:
df["house_direction_clean"].value_counts(dropna=False)

house_direction_clean
<NA>        6949
Đông Nam     698
Tây Bắc      469
Đông Bắc     444
Tây Nam      402
Đông         325
Nam          322
Bắc          231
Tây          160
Name: count, dtype: Int64

In [39]:
direction_price_stats = (
    df.dropna(
        subset=["house_direction_clean"]
    )
    .groupby("house_direction_clean")
    .agg(
        count=("price", "count"),
        mean_price=("price", "mean"),
        mean_price_per_m2=(
            "price_per_m2_million",
            "mean"
        ),
        mean_area=("area", "mean")
    )
    .sort_values(
        "mean_price_per_m2",
        ascending=False
    )
)

direction_price_stats

,count,mean_price,mean_price_per_m2,mean_area
house_direction_clean,,,,
Đông Nam,647,1.485650e+10,118.218351,174.440616
Tây Nam,377,1.275907e+10,113.803921,392.897512
Đông Bắc,417,1.219148e+10,106.471514,180.492950
Tây Bắc,438,1.124920e+10,100.329035,123.787271
Nam,296,1.044024e+10,97.928106,177.290901
Tây,153,9.324688e+09,93.854618,129.545813
Bắc,219,1.169662e+10,87.733633,167.365584
Đông,303,1.290717e+10,85.784986,260.784954


bds hướng Đông Nam có giá trị trung bình /m2 cao nhất 

# 14. Hướng ban công ảnh hưởng thế nào đến giá?

phần nhiều dữ liệu là hướng bancol nó giống như hướng nhà nên nó y chang hướng nhà